# DealerPulse Exploratory Data Analysis

Purpose: fix the metric definitions and confirm the data's story before the UI is built, so the app, the numbers, and DECISIONS.md agree.

Dataset: dealership_data.json. 7 months (Jun 2025 to Dec 2025), 5 branches, 30 reps, 510 leads, 35 targets, 160 deliveries. Synthetic.

### Metric contract
- "Now" is the dataset cutoff 2025-12-31, never the wall clock. Staleness and aging are measured from it.
- Delivery metrics key on delivery_date; lead volume and source key on created_at.
- Revenue is realized only on delivery (deal_value counts once status is delivered).
- The funnel is reconstructed from status_history, not current status.
- Targets are summed only over months in the selected range, with no partial-month proration.
- Anomalies are reported as "Unknown", never dropped.

In [1]:
import json, collections, statistics as st
from datetime import datetime, timezone

DATA_PATH = "../dealership_data.json"   # notebook lives in /analysis
d = json.load(open(DATA_PATH))
LEADS   = d["leads"]
BRANCH  = {b["id"]: b["name"] for b in d["branches"]}
CITY    = {b["id"]: b["city"] for b in d["branches"]}
REPS    = {r["id"]: r for r in d["sales_reps"]}
DELIV   = {x["lead_id"]: x for x in d["deliveries"]}
TARGETS = d["targets"]

CUTOFF = datetime(2025, 12, 31, tzinfo=timezone.utc)      # the analytical "now"
STAGES = ["new","contacted","test_drive","negotiation","order_placed","delivered"]
IDX    = {s:i for i,s in enumerate(STAGES)}
OPEN   = {"new","contacted","test_drive","negotiation","order_placed"}

def parse(s):
    return datetime.fromisoformat(s.replace("Z","+00:00"))

def reached_at(lead, status):
    """First timestamp at which a lead reached `status`, else None."""
    for h in lead["status_history"]:
        if h["status"] == status:
            return parse(h["timestamp"])
    return None

def stage_before_lost(lead):
    prev = "(unknown)"
    for h in lead["status_history"]:
        if h["status"] == "lost":
            return prev
        prev = h["status"]
    return prev

def cr(x):  # rupees -> crore string
    return f"Rs {x/1e7:.2f}cr"

print("Loaded:", len(LEADS), "leads |", len(d["branches"]), "branches |",
      len(REPS), "reps |", len(d["deliveries"]), "deliveries |", len(TARGETS), "targets")

Loaded: 510 leads | 5 branches | 30 reps | 160 deliveries | 35 targets


## 1. Shape and integrity
Validate the data and list the anomalies the UI must handle.

In [2]:
status_dist = collections.Counter(l["status"] for l in LEADS)
print("Current-status distribution:", dict(status_dist))

# Anomaly A: leads marked 'lost' with no closing history event / no lost_reason
lost_no_reason = [l for l in LEADS if l["status"]=="lost" and not l.get("lost_reason")]
lost_no_hist   = [l for l in LEADS if l["status"]=="lost" and
                  (not l["status_history"] or l["status_history"][-1]["status"]!="lost")]
print(f"Anomaly A: 'lost' without lost_reason = {len(lost_no_reason)} ; "
      f"without final 'lost' history event = {len(lost_no_hist)}  -> render as 'Unknown'")

# Anomaly B: delivered leads missing a delivery record, and orphan deliveries
deliv_no_rec = [l for l in LEADS if l["status"]=="delivered" and l["id"] not in DELIV]
lead_ids = {l["id"] for l in LEADS}
orphan_deliv = [x for x in d["deliveries"] if x["lead_id"] not in lead_ids]
print(f"Anomaly B: delivered leads w/o delivery record = {len(deliv_no_rec)} ; "
      f"orphan deliveries = {len(orphan_deliv)}")

# deal_value coverage & stability
print("deal_value present on", sum(1 for l in LEADS if l.get('deal_value')), "of", len(LEADS), "leads")
print("status_history entries carry deal_value:",
      any('deal_value' in h for l in LEADS for h in l['status_history']))

Current-status distribution: {'lost': 288, 'order_placed': 38, 'delivered': 160, 'negotiation': 3, 'test_drive': 6, 'contacted': 10, 'new': 5}
Anomaly A: 'lost' without lost_reason = 14 ; without final 'lost' history event = 14  -> render as 'Unknown'
Anomaly B: delivered leads w/o delivery record = 0 ; orphan deliveries = 0
deal_value present on 510 of 510 leads
status_history entries carry deal_value: False


## 2. Headline: systemic target shortfall
The whole group runs far below target, not one weak branch.

In [3]:
delivered = [l for l in LEADS if l["status"]=="delivered"]
total_target = sum(t["target_units"] for t in TARGETS)
print(f"Overall: {len(delivered)} units delivered vs {total_target} target = "
      f"{100*len(delivered)/total_target:.1f}% attainment  (systemic shortfall)")

# By month (deliveries keyed on delivery_date)
print("\nMonthly units delivered vs target:")
for m in sorted({t['month'] for t in TARGETS}):
    deld = sum(1 for x in d['deliveries'] if x['delivery_date'][:7]==m)
    tgt = sum(t['target_units'] for t in TARGETS if t['month']==m)
    print(f"  {m}  delivered={deld:3}  target={tgt:3}  attain={100*deld/tgt:4.1f}%")

# By branch: conversion + attainment
print("\nBy branch:")
by_b = collections.defaultdict(lambda:[0,0])
for l in LEADS:
    by_b[l['branch_id']][0]+=1
    if l['status']=='delivered': by_b[l['branch_id']][1]+=1
for bid in sorted(by_b):
    tot,dv = by_b[bid]
    tgt = sum(t['target_units'] for t in TARGETS if t['branch_id']==bid)
    print(f"  {BRANCH[bid]:18} ({CITY[bid]:9}) leads={tot:3} delivered={dv:3} "
          f"conv={100*dv/tot:4.1f}%  attain={100*dv/tgt:4.1f}%")

Overall: 160 units delivered vs 1426 target = 11.2% attainment  (systemic shortfall)

Monthly units delivered vs target:
  2025-06  delivered=  0  target=189  attain= 0.0%
  2025-07  delivered= 16  target=183  attain= 8.7%
  2025-08  delivered= 18  target=183  attain= 9.8%
  2025-09  delivered= 24  target=181  attain=13.3%
  2025-10  delivered= 20  target=232  attain= 8.6%
  2025-11  delivered= 30  target=240  attain=12.5%
  2025-12  delivered= 52  target=218  attain=23.9%

By branch:
  Downtown Toyota    (Chennai  ) leads= 97 delivered= 40 conv=41.2%  attain=12.7%
  Highway Toyota     (Chennai  ) leads=109 delivered= 36 conv=33.0%  attain=12.4%
  Lakeside Toyota    (Bangalore) leads= 79 delivered=  6 conv= 7.6%  attain= 2.3%
  Central Toyota     (Hyderabad) leads= 98 delivered= 31 conv=31.6%  attain=12.7%
  Eastside Toyota    (Mumbai   ) leads=127 delivered= 47 conv=37.0%  attain=15.0%


## 3. Conversion funnel
Rebuilt from status_history: of the leads that reached each stage, what share advanced to the next.

In [4]:
reached = collections.Counter()
for l in LEADS:
    for s in set(h["status"] for h in l["status_history"]):
        if s in IDX: reached[s]+=1

print("Funnel (reached each stage) and stage-to-stage progression:")
for i,s in enumerate(STAGES):
    line = f"  {s:12} reached={reached[s]:3}"
    if i+1 < len(STAGES):
        nxt = STAGES[i+1]; adv = reached[nxt]
        line += f"  -> {nxt:12} advance={100*adv/reached[s]:3.0f}%  leak={100-100*adv/reached[s]:3.0f}%"
    print(line)

Funnel (reached each stage) and stage-to-stage progression:
  new          reached=510  -> contacted    advance= 77%  leak= 23%
  contacted    reached=391  -> test_drive   advance= 77%  leak= 23%
  test_drive   reached=300  -> negotiation  advance= 78%  leak= 22%
  negotiation  reached=235  -> order_placed advance= 84%  leak= 16%
  order_placed reached=198  -> delivered    advance= 81%  leak= 19%
  delivered    reached=160


## 4. Losses by stage, value, and reason
Attribute each lost deal to the stage it died at and its deal_value.

In [5]:
loss_cnt = collections.Counter(); loss_val = collections.Counter()
reason_by_stage = collections.defaultdict(collections.Counter)
for l in LEADS:
    if l["status"]=="lost":
        s = stage_before_lost(l)
        loss_cnt[s]+=1; loss_val[s]+=l["deal_value"]
        if l.get("lost_reason"):
            reason_by_stage[s][l["lost_reason"]]+=1

print("Losses by stage-before-lost (count & pipeline value):")
for s,_ in loss_cnt.most_common():
    print(f"  {s:12} losses={loss_cnt[s]:3}  value={cr(loss_val[s])}")

print("\nTop lost reasons by stage:")
for s in ["new","contacted","test_drive","negotiation"]:
    if reason_by_stage[s]:
        print(f"  {s:12}", reason_by_stage[s].most_common(2))

print("\nOverall lost-reason ranking:")
allr = collections.Counter(l['lost_reason'] for l in LEADS
                           if l['status']=='lost' and l.get('lost_reason'))
for r,c in allr.most_common():
    print(f"  {r:32} {c}")

Losses by stage-before-lost (count & pipeline value):
  new          losses=114  value=Rs 27.28cr
  contacted    losses= 81  value=Rs 20.53cr
  test_drive   losses= 59  value=Rs 14.30cr
  negotiation  losses= 34  value=Rs 7.54cr

Top lost reasons by stage:
  new          [('Better offer elsewhere', 18), ('Unresponsive after follow-up', 17)]
  contacted    [('Better offer elsewhere', 13), ('Chose competitor brand', 12)]
  test_drive   [('Budget constraints', 10), ('Financing not approved', 8)]
  negotiation  [('Not ready to purchase', 9), ('Financing not approved', 6)]

Overall lost-reason ranking:
  Better offer elsewhere           40
  Not ready to purchase            40
  Financing not approved           38
  Unresponsive after follow-up     38
  Budget constraints               36
  Chose competitor brand           30
  Dissatisfied with test drive     28
  Relocated to another city        24


## 5. Pipeline velocity
Median time-in-stage from status_history transitions, and the end-to-end sales cycle for delivered leads.

In [6]:
durs = collections.defaultdict(list)
for l in LEADS:
    h = l["status_history"]
    for a,b in zip(h, h[1:]):
        if a["status"] in IDX and b["status"] in IDX and IDX[b["status"]]==IDX[a["status"]]+1:
            durs[a["status"]].append((parse(b["timestamp"])-parse(a["timestamp"])).total_seconds()/86400)

print("Median time-in-stage (days):")
for s in STAGES[:-1]:
    if durs[s]:
        print(f"  {s:12} median={st.median(durs[s]):4.1f}  mean={st.mean(durs[s]):4.1f}  n={len(durs[s])}")

cycle = [ (reached_at(l,'delivered')-parse(l['created_at'])).days
          for l in LEADS if l['status']=='delivered' and reached_at(l,'delivered') ]
cycle.sort()
print(f"\nSales cycle (created -> delivered): median={st.median(cycle)}d  "
      f"mean={st.mean(cycle):.1f}d  p90={cycle[int(0.9*len(cycle))]}d")

# Delivery SLA: delayed vs on-time
delays = [x for x in d['deliveries'] if x.get('delay_reason')]
ontime = [x for x in d['deliveries'] if not x.get('delay_reason')]
print(f"\nDeliveries: {len(delays)} delayed / {len(d['deliveries'])} "
      f"(avg days_to_deliver delayed={st.mean([x['days_to_deliver'] for x in delays]):.1f} "
      f"vs on-time={st.mean([x['days_to_deliver'] for x in ontime]):.1f})")
print("Top delay reasons:", collections.Counter(x['delay_reason'] for x in delays).most_common(4))

Median time-in-stage (days):
  new          median= 1.9  mean= 1.9  n=391
  contacted    median= 5.9  mean= 6.0  n=300
  test_drive   median= 3.8  mean= 4.0  n=235
  negotiation  median= 8.2  mean= 7.9  n=198
  order_placed median=17.0  mean=18.3  n=160

Sales cycle (created -> delivered): median=37.0d  mean=37.6d  p90=50d

Deliveries: 72 delayed / 160 (avg days_to_deliver delayed=25.2 vs on-time=12.7)
Top delay reasons: [('Customer requested date change', 18), ('Logistics delay in transit', 11), ('Vehicle allocation delayed from factory', 11), ('Accessory fitment backlog', 10)]


## 6. Channel and product quality
Weighted by realized revenue per lead, not conversion alone.

In [7]:
sv = collections.defaultdict(lambda:[0,0,0.0])  # leads, delivered, revenue
for l in LEADS:
    sv[l['source']][0]+=1
    if l['status']=='delivered':
        sv[l['source']][1]+=1; sv[l['source']][2]+=l['deal_value']
print("Source quality:")
for s,(n,dc,rev) in sorted(sv.items(), key=lambda x:-x[1][2]/x[1][0]):
    print(f"  {s:14} leads={n:3} conv={100*dc/n:4.1f}%  rev/lead=Rs{rev/n/1e5:4.1f}L  total={cr(rev)}")

mv = collections.defaultdict(lambda:[0,0.0])
for l in LEADS:
    if l['status']=='delivered':
        mv[l['model_interested']][0]+=1; mv[l['model_interested']][1]+=l['deal_value']
print("\nModel revenue concentration (delivered):")
for m,(c,rev) in sorted(mv.items(), key=lambda x:-x[1][1]):
    print(f"  {m:22} units={c:3}  revenue={cr(rev)}")

Source quality:
  walk_in        leads=140 conv=45.7%  rev/lead=Rs11.4L  total=Rs 16.01cr
  auto_expo      leads= 43 conv=30.2%  rev/lead=Rs 8.1L  total=Rs 3.48cr
  website        leads=100 conv=28.0%  rev/lead=Rs 7.0L  total=Rs 6.99cr
  phone_enquiry  leads= 72 conv=27.8%  rev/lead=Rs 6.7L  total=Rs 4.80cr
  referral       leads= 83 conv=30.1%  rev/lead=Rs 6.1L  total=Rs 5.09cr
  social_media   leads= 72 conv=13.9%  rev/lead=Rs 3.5L  total=Rs 2.51cr

Model revenue concentration (delivered):
  Fortuner               units= 30  revenue=Rs 12.61cr
  Innova Hycross         units= 28  revenue=Rs 7.19cr
  Camry                  units= 10  revenue=Rs 5.34cr
  Innova Crysta          units= 17  revenue=Rs 4.22cr
  Urban Cruiser Hyryder  units= 27  revenue=Rs 4.05cr
  Glanza                 units= 44  revenue=Rs 3.97cr
  Hilux                  units=  4  revenue=Rs 1.49cr


## 7. Action Center alert rules
Every rule measures staleness from the cutoff, is explainable, and resolves to the affected leads with owner and value.

In [8]:
def days_stale(l):
    return (CUTOFF - parse(l['last_activity_at'])).days

open_leads = [l for l in LEADS if l['status'] in OPEN]
cold = [l for l in open_leads if days_stale(l) >= 7]
op_stale = [l for l in open_leads if l['status']=='order_placed' and days_stale(l) >= 7]
overdue  = [l for l in open_leads if l.get('expected_close_date')
            and parse(l['expected_close_date']+'T00:00:00Z') < CUTOFF]

print(f"Open pipeline           : {len(open_leads):3} leads  value={cr(sum(l['deal_value'] for l in open_leads))}")
print(f"Cold (>=7d no activity) : {len(cold):3} leads  value={cr(sum(l['deal_value'] for l in cold))}")
print(f"Order placed but stale  : {len(op_stale):3} leads  value={cr(sum(l['deal_value'] for l in op_stale))}  <- highest urgency")
print(f"Past expected close date: {len(overdue):3} leads  value={cr(sum(l['deal_value'] for l in overdue))}")

# Simple at-risk score = value(cr) * stage_depth * log-ish age; illustrative ranking
def score(l):
    return (l['deal_value']/1e7) * (IDX.get(l['status'],0)+1) * max(days_stale(l),1)
top = sorted(open_leads, key=score, reverse=True)[:8]
print("\nTop at-risk leads (value x stage-depth x staleness):")
for l in top:
    print(f"  {l['id']} {l['customer_name']:18} {BRANCH[l['branch_id']]:16} "
          f"{l['status']:12} stale={days_stale(l):2}d  value={cr(l['deal_value'])}")

Open pipeline           :  62 leads  value=Rs 15.15cr
Cold (>=7d no activity) :  35 leads  value=Rs 8.07cr
Order placed but stale  :  32 leads  value=Rs 7.58cr  <- highest urgency
Past expected close date:  30 leads  value=Rs 6.80cr

Top at-risk leads (value x stage-depth x staleness):
  L0022 Omkar Varma        Lakeside Toyota  order_placed stale=194d  value=Rs 0.51cr
  L0009 Rakesh Naidu       Central Toyota   order_placed stale=175d  value=Rs 0.29cr
  L0180 Lata Srinivasan    Highway Toyota   order_placed stale=124d  value=Rs 0.25cr
  L0010 Manju Srinivasan   Highway Toyota   order_placed stale=186d  value=Rs 0.14cr
  L0296 Nalini Prasad      Downtown Toyota  order_placed stale=54d  value=Rs 0.46cr
  L0321 Ishaan Yadav       Downtown Toyota  order_placed stale=46d  value=Rs 0.51cr
  L0260 Bharat Zaveri      Eastside Toyota  order_placed stale=67d  value=Rs 0.29cr
  L0042 Gautam Iyengar     Highway Toyota   order_placed stale=180d  value=Rs 0.10cr


## 8. Rep and branch comparison
Lakeside is the operational outlier. Several of its reps have thin lead counts, so rankings show sample size and apply a 5-lead floor.

In [9]:
rp = collections.defaultdict(lambda:[0,0,0.0])  # leads, delivered, revenue
for l in LEADS:
    rp[l['assigned_to']][0]+=1
    if l['status']=='delivered':
        rp[l['assigned_to']][1]+=1; rp[l['assigned_to']][2]+=l['deal_value']

rows = []
for rid,(n,dc,rev) in rp.items():
    if n >= 5:  # min-sample floor
        rows.append((100*dc/n, n, dc, rev, rid))
rows.sort()
print("Lowest-converting reps (>=5 leads):")
for conv,n,dc,rev,rid in rows[:5]:
    print(f"  {REPS[rid]['name']:18} {BRANCH[REPS[rid]['branch_id']]:16} "
          f"conv={conv:4.1f}%  leads={n:3}  revenue={cr(rev)}")
print("\nHighest-converting reps (>=5 leads):")
for conv,n,dc,rev,rid in rows[-5:][::-1]:
    print(f"  {REPS[rid]['name']:18} {BRANCH[REPS[rid]['branch_id']]:16} "
          f"conv={conv:4.1f}%  leads={n:3}  revenue={cr(rev)}")

Lowest-converting reps (>=5 leads):
  Venkat Mishra      Lakeside Toyota  conv= 4.5%  leads= 22  revenue=Rs 0.10cr
  Revathi Pandey     Lakeside Toyota  conv= 7.1%  leads= 14  revenue=Rs 0.08cr
  Kavitha Joshi      Lakeside Toyota  conv= 7.7%  leads= 13  revenue=Rs 0.12cr
  Vikram Patel       Lakeside Toyota  conv= 8.3%  leads= 12  revenue=Rs 0.12cr
  Sanjay Rao         Lakeside Toyota  conv=11.1%  leads= 18  revenue=Rs 0.64cr

Highest-converting reps (>=5 leads):
  Priya Choudhury    Downtown Toyota  conv=57.1%  leads= 14  revenue=Rs 2.83cr
  Suresh Nair        Highway Toyota   conv=50.0%  leads= 22  revenue=Rs 2.16cr
  Sanjay Kulkarni    Eastside Toyota  conv=48.0%  leads= 25  revenue=Rs 2.98cr
  Prakash Gupta      Central Toyota   conv=42.9%  leads= 28  revenue=Rs 2.18cr
  Ananya Pandey      Downtown Toyota  conv=42.1%  leads= 19  revenue=Rs 1.92cr


## 9. Traps tested and rejected
What was not built, and why.

In [10]:
# Trap 1: speed-to-lead (new -> contacted) does NOT predict conversion here
resp = []
for l in LEADS:
    tn, tc = reached_at(l,'new'), reached_at(l,'contacted')
    if tn and tc:
        resp.append(((tc-tn).total_seconds()/3600, l['status']=='delivered'))
print("Speed-to-lead vs conversion (noisy / no clean signal):")
for lo,hi,lbl in [(0,24,'<24h'),(24,72,'1-3d'),(72,1e9,'>3d')]:
    g=[d_ for h,d_ in resp if lo<=h<hi]
    if g: print(f"  {lbl:5} n={len(g):3} conv={100*sum(g)/len(g):4.1f}%")

# Trap 2: 'touch count' is circular -- delivered leads simply have all 6 stage entries
print("\n'Touch count' vs conversion (circular -- do NOT use as a predictor):")
tc = collections.defaultdict(lambda:[0,0])
for l in LEADS:
    n=len(l['status_history']); tc[n][0]+=1
    if l['status']=='delivered': tc[n][1]+=1
for n in sorted(tc):
    t,c=tc[n]; print(f"  {n} entries: n={t:3} conv={100*c/t:3.0f}%")

Speed-to-lead vs conversion (noisy / no clean signal):
  <24h  n= 45 conv=37.8%
  1-3d  n=321 conv=39.9%
  >3d   n= 25 conv=60.0%

'Touch count' vs conversion (circular -- do NOT use as a predictor):
  1 entries: n=  7 conv=  0%
  2 entries: n=128 conv=  0%
  3 entries: n= 85 conv=  0%
  4 entries: n= 60 conv=  0%
  5 entries: n= 70 conv=  0%
  6 entries: n=160 conv=100%


## Key takeaways for DECISIONS.md
1. Systemic shortfall: 160/1426 units = 11.2% overall attainment; the best month (Dec) reaches only 23.9%. Red numbers are a diagnosis, not a bug.
2. Lakeside is the outlier: 7.6% conversion vs 33-41% elsewhere; its reps fill the bottom of the leaderboard.
3. Biggest leak is early and high-value: 114 leads lost at `new` = ~Rs27cr never truly engaged. 68% of the 288 losses die at `new` or `contacted`.
4. Loss reasons are dominated by early-stage churn (better offer, unresponsive, budget). 'Financing not approved' recurs at every stage (14/10/8/6 from new to negotiation), a standing finance-desk friction, not a test-drive/negotiation cluster.
5. Channel quality: walk-ins deliver about half of all revenue at 45.7% conversion; social media is weakest by conversion and revenue per lead.
6. Revenue concentration: 3 models drive 65% of the Rs38.9cr delivered (Fortuner 32%, Innova Hycross 19%, Camry 14%).
7. Actionable now (as of 2025-12-31): 35 cold leads, 32 stale order-placed (~Rs7.6cr), 30 past expected close (~Rs6.8cr).
8. Anomalies handled: 14 malformed 'lost' records surfaced as 'Unknown'.
9. Deliberately not built: forecasting (pipeline too small vs targets), speed-to-lead and touch-count predictors (no real or circular signal).